# UCI HAR ZARA Inference

Run the UCI example with a single retrieval index. This notebook illustrates the non-RRF retrieval setting.


In [1]:
import os
from collections import Counter
import pickle
import pandas as pd
import random
from mantis.trainer import MantisTrainer
from mantis.architecture import Mantis8M
import numpy as np
from collections import defaultdict
import torch
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

output_path = "./dataset/uci"

all_database_embeddings = np.load(os.path.join(output_path, "all_database_embeddings.npy"))
print("all_database_embeddings shape:", all_database_embeddings.shape)

with open(os.path.join(output_path, 'uci_database_segments.pkl'), 'rb') as f:
    all_database_segments = pickle.load(f)

with open(os.path.join(output_path, f'uci_test_data.pkl'), 'rb') as f:
    all_test_segments = pickle.load(f)

with open(os.path.join(output_path, 'uci_database_labels.pkl'), 'rb') as f:
    all_database_labels = pickle.load(f)

with open(os.path.join(output_path, f'uci_test_labels.pkl'), 'rb') as f:
    all_test_labels = pickle.load(f)

from collections import defaultdict, Counter

activity_subject_counts = defaultdict(Counter)
for lbl in all_test_labels:
    act = lbl["activity"]
    subj = lbl["subject"]
    activity_subject_counts[act][subj] += 1

for act, subj_counts in activity_subject_counts.items():
    print(f"Activity = {act}")
    for subj, cnt in subj_counts.items():
        print(f"    Subject {subj}: {cnt}")
    print()


all_database_embeddings shape: (7352, 1536)
Activity = 4
    Subject 10: 5
    Subject 18: 5
    Subject 20: 5
    Subject 12: 5
    Subject 24: 4
    Subject 9: 4
    Subject 13: 4
    Subject 2: 4
    Subject 4: 4

Activity = 3
    Subject 4: 5
    Subject 2: 5
    Subject 13: 5
    Subject 10: 5
    Subject 20: 4
    Subject 24: 4
    Subject 12: 4
    Subject 9: 4
    Subject 18: 4

Activity = 5
    Subject 4: 5
    Subject 9: 5
    Subject 20: 5
    Subject 24: 5
    Subject 10: 4
    Subject 12: 4
    Subject 13: 4
    Subject 2: 4
    Subject 18: 4

Activity = 0
    Subject 18: 5
    Subject 4: 5
    Subject 12: 5
    Subject 10: 5
    Subject 24: 4
    Subject 20: 4
    Subject 13: 4
    Subject 2: 4
    Subject 9: 4

Activity = 2
    Subject 4: 5
    Subject 12: 5
    Subject 13: 5
    Subject 2: 5
    Subject 10: 4
    Subject 18: 4
    Subject 24: 4
    Subject 20: 4
    Subject 9: 4

Activity = 1
    Subject 12: 5
    Subject 13: 5
    Subject 24: 5
    Subject 9: 5
    Sub

## Load Time-Series Encoder

Load the frozen Mantis encoder for query embedding.


In [2]:
network = Mantis8M(device='mps')
network = network.from_pretrained("paris-noah/Mantis-8M")

model = MantisTrainer(device='mps', network=network)


## Build Retrieval Index

Build a single FAISS index over UCI database embeddings.


In [3]:
import faiss

# Build the FAISS database index.
all_database_embeddings = all_database_embeddings.astype(np.float32)
faiss.normalize_L2(all_database_embeddings)

dim = all_database_embeddings.shape[1]
index_sensor = faiss.IndexFlatIP(dim)
index_sensor.add(all_database_embeddings)


## Retrieval Helpers

Define FAISS retrieval and retrieval-quality inspection helpers.


In [4]:
def query_faiss(index, query_embedding, top_k=5000, sim_diff=0.15):
    """Search a FAISS index and keep candidates within sim_diff of the top match."""
    query_embedding = query_embedding.astype(np.float32)
    faiss.normalize_L2(query_embedding)

    D, I = index.search(query_embedding, top_k)
    D = D.squeeze(0)
    I = I.squeeze(0)

    top1 = D[0]
    threshold = top1 - sim_diff

    mask = D >= threshold

    return D[mask], I[mask]


## Activity Labels

Define the UCI label mapping used in prompts and evaluation.


In [5]:
activity_map = {
    1: 'Walking',
    2: 'Walking upstairs',
    3: 'Walking downstairs',
    4: 'Sitting',
    5: 'Standing',
    6: 'Laying'
}
label2id = {}
id2label = {}
for i, (key, value) in enumerate(activity_map.items()):
    label2id[value]=i
    id2label[i]=value
print(f"label2id:\n{label2id}")
print(f"id2label:\n{id2label}")


label2id:
{'Walking': 0, 'Walking upstairs': 1, 'Walking downstairs': 2, 'Sitting': 3, 'Standing': 4, 'Laying': 5}
id2label:
{0: 'Walking', 1: 'Walking upstairs', 2: 'Walking downstairs', 3: 'Sitting', 4: 'Standing', 5: 'Laying'}


## Feature Constants And Utilities

Set signal-processing constants and basic helper functions.


In [8]:
import torch
import math
from typing import Dict
from scipy.signal import stft
import math
from scipy.stats import iqr, skew, kurtosis, entropy
from scipy.signal import welch, detrend, correlate

# Constants
FS = 50                      # Hz
DT = 1.0 / FS
SEQ_LEN = 128
ORDER=4

## Feature Extraction

Define the handcrafted feature extractor used to compare query and retrieved examples.


In [9]:
from scipy.integrate import cumulative_trapezoid
from scipy.signal import butter, filtfilt
from spectrum import arburg
from scipy.spatial.distance import pdist, squareform
import pywt
import numpy as np
import math
import itertools
from collections import Counter

def spectral_centroid(freqs, psd):
    """Compute the power-weighted average frequency of a spectrum."""
    return np.sum(freqs * psd) / (np.sum(psd) + 1e-12)

def compute_acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    """Compute a normalized autocorrelation sequence up to a maximum lag."""
    x = x - x.mean()
    N = len(x)
    max_lag = min(max_lag, N-1)

    if np.all(x == 0) or np.var(x) < 1e-8:
        acf = np.zeros(max_lag+1, dtype=float)
        acf[0] = 1.0
        return acf

    r = correlate(x, x, mode='full')
    r = r[N-1 : N-1 + max_lag + 1]

    denom = r[0] if abs(r[0]) > 1e-12 else 1e-12
    acf = r / denom

    acf[0] = 1.0

    return acf

def zero_cross_centered(channel_data):
    """Count zero crossings after removing the signal mean."""
    centered = channel_data - np.mean(channel_data)
    signs = np.sign(centered)
    return np.sum(signs[:-1] * signs[1:] < 0)

def band_power(psd_freq, psd_val, fmin, fmax):
    """Integrate spectral power over a frequency band."""
    idx = np.logical_and(psd_freq >= fmin, psd_freq <= fmax)
    return np.trapz(psd_val[idx], psd_freq[idx])

def lowpass(data, cutoff=0.3, fs=FS, order=3):
    """Keep low-frequency components, which mainly capture gravity."""
    b, a = butter(order, cutoff/(fs/2), btype='low')
    return filtfilt(b, a, data)

def highpass(data, cutoff=0.3, fs=FS, order=3):
    """Remove low-frequency gravity drift and keep dynamic motion."""
    b, a = butter(order, cutoff/(fs/2), btype='high')
    return filtfilt(b, a, data)

def channel_corr(data):
    """Compute within-placement correlations between accelerometer and gyroscope channels."""
    feats = {}

    acc = data[0:3, :]  * 9.80665                            # (3, seq_len)
    gyr = data[3:6, :]                   # (3, seq_len)

    channel_names = [
        "acc_x", "acc_y", "acc_z",
        "gyro_x", "gyro_y", "gyro_z"
    ]

    centered = data - data.mean(axis=1, keepdims=True)   # shape = (6, seq_len)

    corr_mat = np.corrcoef(centered)

    for i in range(6):
        for j in range(i + 1, 6):
            key = f"corr_{channel_names[i]}_{channel_names[j]}"
            feats[key] = float(corr_mat[i, j])

    acc_mag  = np.linalg.norm(acc, axis=0)        # shape (seq_len,)
    gyro_mag = np.linalg.norm(gyr, axis=0)

    acc_mag_c  = acc_mag  - acc_mag.mean()
    gyro_mag_c = gyro_mag - gyro_mag.mean()

    if acc_mag_c.std() > 1e-8 and gyro_mag_c.std() > 1e-8:
        corr_mag = np.corrcoef(acc_mag_c, gyro_mag_c)[0, 1]
    else:
        corr_mag = 0.0

    feats["corr_acc_mag_gyro_mag"] = float(corr_mag)

    return feats

def auto_reg_berg(channel_data, name, axis, order=4):
    """Estimate autoregressive coefficients with Burg's method."""
    feats = {}
    ar_coeffs, variance, _ = arburg(channel_data, order)
    n = len(ar_coeffs) - 1

    for k in range(1, n+1):
        feats[f"{name}_{axis}_ar{k}"] = float(ar_coeffs[k])
    for k in range(n+1, order+1):
        feats[f"{name}_{axis}_ar{k}"] = 0.0
    feats[f"{name}_{axis}_ar_var"] = float(variance)
    return feats

def recurrence_rate(channel_data, name, axis, m=2, tau=1, eps=None, exclude_diag=True):
    """Compute the recurrence rate for a one-dimensional signal."""
    feats = {}

    L = len(channel_data)
    max_m = (L - 1) // tau + 1
    m_use = min(m, max_m)

    N_embed = L - (m_use - 1) * tau

    X = np.column_stack([channel_data[j * tau : j * tau + N_embed] for j in range(m_use)])
    D = squareform(pdist(X, metric='euclidean'))

    if eps is None:
        eps = 0.1 * np.std(channel_data)
    R = (D <= eps).astype(int)

    if exclude_diag:
        total = N_embed * (N_embed - 1)
        feats[f"{name}_{axis}_rr"] = (R.sum() - N_embed) / total
    else:
        total = N_embed * N_embed
        feats[f"{name}_{axis}_rr"] = R.sum() / total

    return feats

def wavelet_decomposition(channel_data, name, axis, maxlevel=5):
    """Extract wavelet packet energy, entropy, and coefficient statistics."""
    feats = {}
    wp = pywt.WaveletPacket(data=channel_data,
                         wavelet='db4',
                         mode='symmetric',
                         maxlevel=maxlevel)

    for lvl in range(1, maxlevel+1):
        nodes = wp.get_level(lvl, order='freq')
        coeffs = np.hstack([n.data for n in nodes])

        sum_abs = np.sum(np.abs(coeffs))
        feats[f"{name}_{axis}_wpd_L{lvl}_sum"] = round(sum_abs, 6)

        energy = np.sum(coeffs**2)
        feats[f"{name}_{axis}_wpd_L{lvl}_energy"] = round(energy, 6)

        psq = coeffs**2
        p = psq / (np.sum(psq) + 1e-12)
        entropy = -np.sum(p * np.log2(p + 1e-12))
        feats[f"{name}_{axis}_wpd_L{lvl}_entropy"] = round(entropy, 6)

    return feats

def permutation_entropy(channel_data, name, axis, m, tau=1, normalized=False):
    """Compute ordinal-pattern permutation entropy for a signal channel."""
    feats = {}

    N = len(channel_data)
    n_windows = N - (m - 1) * tau
    if n_windows <= 0:
        raise ValueError("Sequence is too short for the requested embedding; reduce m or tau.")

    perms = list(itertools.permutations(range(m)))
    perm_counts = Counter()

    for i in range(n_windows):
        window = channel_data[i : i + (m - 1) * tau + 1 : tau]
        pattern = tuple(np.argsort(window))
        perm_counts[pattern] += 1

    counts = np.array([perm_counts[p] for p in perms], dtype=float)
    probs = counts / counts.sum()
    probs = probs[probs > 0]

    pe = -np.sum(probs * np.log(probs))

    if normalized:
        pe /= math.log(math.factorial(m))

    feats[f"{name}_{axis}_pe"] = float(pe)

    return feats

def extract_time_domain_features(channel_data, name, axis):
    """Extract summary statistics from one time-domain signal channel."""
    feats: Dict[str,float] = {}
    seq_len = channel_data.shape[0]

    diff = np.diff(channel_data)

    # time-domain
    mean   = np.mean(channel_data)
    std    = np.std(channel_data)
    maxv   = np.max(channel_data)
    minv   = np.min(channel_data)
    med    = np.median(channel_data)

    rms        = np.sqrt(np.mean(channel_data**2))
    peak       = np.max(np.abs(channel_data))
    var        = np.var(channel_data)
    mav = np.mean(np.abs(channel_data))
    sms = np.sum(np.abs(channel_data)) / seq_len

    # slope
    t = np.arange(seq_len, dtype=channel_data.dtype)
    t_mean, channel_data_mean = t.mean(), mean
    num = ((t - t_mean) * (channel_data - channel_data_mean)).sum()
    den = ((t - t_mean)**2).sum()
    slope = num / den if den != 0 else 0.0

    zero_crossings = zero_cross_centered(channel_data)
    zc_rate = zero_crossings / (seq_len - 1)

    diff_mean  = np.mean(diff) if diff.size>0 else 0.0
    diff_rms   = np.sqrt(np.mean(diff**2)) if diff.size>0 else 0.0
    diff_std   = np.std(diff) if diff.size>0 else 0.0

    prange = maxv - minv
    total = np.sum(channel_data)
    total_abs = np.sum(np.abs(channel_data))

    iqr_v  = iqr(channel_data)
    skew_v = skew(channel_data) if std > 0 else 0.0
    kurt_v = kurtosis(channel_data, fisher=False) if std > 0 else 0.0

    feats[f"{name}_{axis}_mean"]     = mean
    feats[f"{name}_{axis}_std"]      = std
    feats[f"{name}_{axis}_max"]      = maxv
    feats[f"{name}_{axis}_min"]      = minv
    feats[f"{name}_{axis}_median"]   = med

    feats[f"{name}_{axis}_rms"]       = rms
    feats[f"{name}_{axis}_peak"]      = peak
    feats[f"{name}_{axis}_var"]       = var

    feats[f"{name}_{axis}_zc_rate"]      = zc_rate

    feats[f"{name}_{axis}_slope"]     = slope
    feats[f"{name}_{axis}_diff_mean"] = diff_mean
    feats[f"{name}_{axis}_diff_rms"]  = diff_rms
    feats[f"{name}_{axis}_diff_std"]  = diff_std

    feats[f"{name}_{axis}_range"]    = prange
    feats[f"{name}_{axis}_sum"]       = total

    feats[f"{name}_{axis}_sav"]   = total_abs
    feats[f"{name}_{axis}_mav"] = mav

    feats[f"{name}_{axis}_iqr"]      = iqr_v
    feats[f"{name}_{axis}_skew"]     = skew_v
    feats[f"{name}_{axis}_kurtosis"] = kurt_v

    feats[f"{name}_{axis}_sma"]  = sms

    return feats

def extract_frequency_domain_features(channel_data, name, axis):
    """Extract power-spectrum statistics from one signal channel."""
    feat = {}

     # 1. PSD
    # sig_detrend = detrend(channel_data)
    signal_len = channel_data.shape[0]
    nperseg = min(256, signal_len)
    nfft = max(256, 2**int(np.ceil(np.log2(nperseg))))
    freqs, psd = welch(channel_data, fs=FS, nperseg=nperseg, nfft=nfft, detrend=False)

    # 2. Band power (low/mid/high)
    bands = {"low":(0, 0.5), "mid":(0.5, 3.0), "high":(3.0, min(15.0, freqs[-1]))}
    total_power = np.trapz(psd, freqs) + 1e-8
    for band_name, (low, high) in bands.items():
        if low >= freqs[-1]:
            continue
        bp = band_power(freqs, psd, low, high)
        feat[f"{name}_{axis}_bp_fft_{band_name}"] = bp
        feat[f"{name}_{axis}_bp_fft_{band_name}_ratio"] = bp / total_power

    # 3. Dominant freq & peak
    dom_idx = np.argmax(psd[1:]) + 1
    feat[f"{name}_{axis}_fft_dom_freq"]  = freqs[dom_idx]
    feat[f"{name}_{axis}_fft_dom_power"] = psd[dom_idx]

    peaks = np.where((psd[1:-1] > psd[:-2]) & (psd[1:-1] > psd[2:]))[0] + 1
    if peaks.size>1:
        second = peaks[np.argsort(psd[peaks])[-2]]
        feat[f"{name}_{axis}_fft_2nd_peak_freq"]  = freqs[second]
        feat[f"{name}_{axis}_fft_2nd_peak_power"] = psd[second]

    # 5. Spectral Centroid
    cent = spectral_centroid(freqs, psd)
    feat[f"{name}_{axis}_fft_sp_centroid"] = cent

    # 6. Spectral Entropy & Flatness
    p_norm = psd/np.sum(psd)
    feat[f"{name}_{axis}_fft_sp_entropy"]  = entropy(p_norm)

    feat[f"{name}_{axis}_fft_skew"] = skew(psd)

    feat[f"{name}_{axis}_fft_kurtosis"] = kurtosis(psd)

    ws = np.sum(psd) + 1e-8
    weighted_freq = np.sum(freqs * psd) / ws
    feat[f"{name}_{axis}_fft_weighted_avg_freq"] = weighted_freq

    energy = np.trapz(psd**2, freqs)
    feat[f"{name}_{axis}_fft_energy"] = energy

    max_idx = int(np.argmax(psd))
    feat[f"{name}_{axis}_fft_max_idx"] = max_idx

    return feat

def extract_stft_features(channel_data, name, axis):
    """Extract summary statistics from the short-time Fourier transform."""
    signal_len = channel_data.shape[0]
    nperseg = min(128, signal_len)
    noverlap = nperseg // 2
    nfft = 2 ** int(np.ceil(np.log2(nperseg)))

    f, t, Z = stft(channel_data, fs=FS, nperseg=nperseg, noverlap=noverlap, nfft=nfft, detrend=False)
    M = np.abs(Z)  # magnitude spectrogram, shape=(len(f), len(t))
    psd = M**2

    bands = {"low":(0.0, 0.5), "mid":(0.5,3.0), "high":(3.0, min(15.0, FS/2-0.1))}
    feats = {}

    for band_name, (low, high) in bands.items():
        mask = (f >= low) & (f < high)
        frame_energy = psd[mask,:].sum(axis=0)      # shape (T,)
        feats[f'{name}_{axis}_stft_{band_name}_max']  = frame_energy.max()
        feats[f'{name}_{axis}_stft_{band_name}_mean'] = frame_energy.mean()
        feats[f'{name}_{axis}_stft_{band_name}_std']  = frame_energy.std()

    p_norm = psd / (psd.sum(axis=0, keepdims=True)+1e-8)
    ent = -np.sum(p_norm * np.log(p_norm+1e-12), axis=0)  # per-frame entropy
    feats[f"{name}_{axis}_stft_ent_mean"] = ent.mean()
    feats[f"{name}_{axis}_stft_ent_max"] = ent.max()
    feats[f"{name}_{axis}_stft_ent_std"]  = ent.std()

    # per-frame centroid
    cent = np.sum(f[:,None] * psd, axis=0) / (psd.sum(axis=0)+1e-8)
    feats[f"{name}_{axis}_stft_centroid_mean"] = cent.mean()
    feats[f"{name}_{axis}_stft_centroid_max"]  = cent.max()
    feats[f"{name}_{axis}_stft_centroid_std"]  = cent.std()

    return feats

def extract_key_acf_features(channel_data, name, axis, max_lag = 100) -> dict:
    """Extract the first local ACF peak, valley, and zero-crossing lag."""
    feats = {}
    acf = compute_acf(channel_data, max_lag)

    first_peak = None
    for k in range(1, max_lag):
        if acf[k] > acf[k-1] and acf[k] > acf[k+1]:
            first_peak = k
            break
    feats[f"{name}_{axis}_acf_first_peak_lag"] = first_peak if first_peak is not None else -1.0

    first_min = None
    for k in range(1, max_lag):
        if acf[k] < acf[k-1] and acf[k] < acf[k+1]:
            first_min = k
            break
    feats[f"{name}_{axis}_acf_first_min_lag"] = first_min if first_min is not None else -1.0

    first_zero = None
    for k in range(1, max_lag+1):
        if acf[k] <= 0 and acf[k-1] > 0:
            denominator = acf[k-1] - acf[k]
            if abs(denominator) < 1e-12:
                frac = 0.5
            else:
                frac = acf[k-1] / denominator
            first_zero = (k - 1) + frac
            break
    feats[f"{name}_{axis}_acf_first_zero_lag"] = first_zero if first_zero is not None else -1.0

    return feats

def extract_jerk_features(channel_data, name, axis):
    """Extract statistics from the first derivative of a signal channel."""
    jerk = np.diff(channel_data) / DT
    feats = {}

    if np.all(jerk == 0):
        return {
            f"{name}_{axis}_jerk_rms": 0.0,
            f"{name}_{axis}_jerk_peak": 0.0,
            f"{name}_{axis}_jerk_zc_rate": 0.0
        }

    # 2. RMS & Peak
    rms   = np.sqrt(np.mean(jerk**2) + 1e-12)
    peak  = np.max(np.abs(jerk))

    signs        = np.sign(jerk)
    nonzero_mask = signs != 0
    filtered     = signs[nonzero_mask]
    if len(filtered) < 2:
        zc_rate = 0.0
    else:
        zc      = np.sum(filtered[:-1] != filtered[1:])
        zc_rate = zc / (len(filtered)-1)

    feats[f"{name}_{axis}_jerk_rms"]     = round(rms, 4)
    feats[f"{name}_{axis}_jerk_peak"]    = round(peak, 4)
    feats[f"{name}_{axis}_jerk_zc_rate"] = round(zc_rate, 4)
    return feats

def extract_features(data):
    """
    data_: (6, seq_len) numpy.ndarray = [acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z]
    returns: dict of rounded features
    """
    seq_len = data.shape[-1]
    assert seq_len==SEQ_LEN

    # split out and convert gyro to rad/s
    acc = data[0:3, :] * 9.80665                     # (3, seq_len)
    gyr = data[3:6, :]                   # (3, seq_len)

    # prepare common constants
    sensors = {"acc": acc, "gyro": gyr}
    feats: Dict[str,float] = {}

    for name, sensor in sensors.items():
        # per-axis features

        # mag = torch.linalg.vector_norm(sensor, dim=0)       # (seq_len,)
        mag = np.linalg.norm(sensor, axis=0)        # (seq_len,)

        for idx, axis in enumerate(("x","y","z")):
            channel_data = sensor[idx]          # 1D array, length seq_len

            # time-domain
            feats.update(extract_time_domain_features(channel_data, name, axis))

            feats.update(extract_frequency_domain_features(channel_data, name, axis))
            feats.update(extract_stft_features(channel_data, name, axis))
            feats.update(extract_key_acf_features(channel_data, name, axis, min(FS, seq_len//2)))
            feats.update(extract_jerk_features(channel_data, name, axis))
            feats.update(auto_reg_berg(channel_data, name, axis, ORDER))
            feats.update(recurrence_rate(channel_data, name, axis, m=2, tau=1, eps=None, exclude_diag=True))
            feats.update(wavelet_decomposition(channel_data, name, axis, maxlevel=5))
            feats.update(permutation_entropy(channel_data, name, axis, m=3, tau=1, normalized=False))

        feats.update(extract_time_domain_features(mag, name, "mag"))
        feats.update(extract_frequency_domain_features(mag, name, "mag"))
        feats.update(extract_stft_features(mag, name, "mag"))
        feats.update(extract_key_acf_features(mag, name, "mag", min(FS, seq_len//2)))
        feats.update(extract_jerk_features(mag, name, "mag"))
        feats.update(auto_reg_berg(mag, name, "mag", ORDER))
        feats.update(recurrence_rate(mag, name, "mag", m=2, tau=1, eps=None, exclude_diag=True))
        feats.update(wavelet_decomposition(mag, name, "mag", maxlevel=5))
        feats.update(permutation_entropy(mag, name, "mag", m=3, tau=1, normalized=True))
        # if name == "acc":
        #     feats.update(extract_velocity_features(mag, name, "mag", fs=FS))
    g_x = lowpass(acc[0], cutoff=0.1)
    g_y = lowpass(acc[1], cutoff=0.1)
    g_z = lowpass(acc[2], cutoff=0.1)
    g_mag = np.sqrt(g_x**2 + g_y**2 + g_z**2) + 1e-8
    theta_g = np.arccos(np.clip(g_z / g_mag, -1, 1))
    feats[f"acc_z_grav_angle_mean"] = float(np.mean(theta_g))
    feats[f"acc_z_grav_angle_std"] = float(np.std(theta_g))

    a_dyn_x = highpass(acc[0], cutoff=0.05)
    a_dyn_y = highpass(acc[1], cutoff=0.05)
    a_dyn_z = highpass(acc[2], cutoff=0.05)
    mag_dyn = np.sqrt(a_dyn_x**2 + a_dyn_y**2 + a_dyn_z**2) + 1e-8
    theta_dyn = np.arccos(np.clip(a_dyn_z / mag_dyn, -1, 1))
    feats[f"acc_z_dyn_angle_mean"] = float(np.mean(theta_dyn))
    feats[f"acc_z_dyn_angle_std"] = float(np.std(theta_dyn))
    feats[f"acc_z_dyn_sign"] = float(np.sign(np.mean(a_dyn_z)))

    # 3) cross-sensor ratio
    feats["intensity_ratio_acc_gyro"] = feats["acc_mag_rms"] / (feats["gyro_mag_rms"] + 1e-9)

    feats.update(channel_corr(data))

    # for k,v in list(feats.items()):
    #     feats[k] = _smart_round(k, v)
    return feats


## Load Gemini And Feature Knowledge

Load Gemini utilities, configure the client from `GOOGLE_API_KEY`, and load the saved UCI pairwise feature-importance knowledge base.


In [ ]:
from google import genai
from google.genai import types
import re, json
from itertools import combinations

all_database_features = []
for seg, y in zip(all_database_segments, all_database_labels):
    feats = extract_features(seg.transpose(1, 0))
    feats['activity'] = y['activity']
    feats['user_id']  = y['subject']
    all_database_features.append(feats)

all_database_features_df = pd.DataFrame(all_database_features)


## Configure Gemini Client

Read the Gemini API key from the environment and create the client.


In [16]:
api_key = os.environ.get("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("Set GOOGLE_API_KEY before running this notebook.")
client = genai.Client(api_key=api_key)

with open("./dataset/uci2/activity_pair_f1_macro_feature_importance.json", "r") as fp:
    nested = json.load(fp)

# Flatten nested JSON to (activity_a, activity_b) -> feature-importance dict.
pair_map = {}
for a, inner in nested.items():
    for b, fmap in inner.items():
        pair_map[(a, b)] = fmap


## Feature Glossary

Define the feature glossary subset helper and full glossary used in prompts.


In [17]:
from typing import List, Dict

def extract_glossary_for_features(
    feature_names: List[str],
    full_glossary: Dict[str, str]
) -> Dict[str, str]:
    """Return glossary entries whose tokens appear in any selected feature name."""
    selected = {}
    tokens = set()
    for name in feature_names:
        for tk in name.split('_'):
            if tk in full_glossary:
                tokens.add(tk)
    for tk in tokens:
        selected[tk] = full_glossary[tk]
    return selected

full_glossary = {
    "acc": "accelerometer",
    "gyro": "gyroscope",
    "x": "sensor x-axis",
    "y": "sensor y-axis",
    "z": "sensor z-axis",
    "mag": "vector magnitude, i.e. √(x²+y²+z²)",
    "mean": "average over the window",
    "std": "standard deviation",
    "iqr": "interquartile range (75th–25th percentile)",
    "max": "maximum value over the window",
    "median": "median value",
    "min": "minimum value",
    "peak": "maximum of the absolute sensor signal",
    "rms": "root mean square",
    "bp": "band power in a frequency band",
    "low": "frequency band roughly 0–0.5 Hz",
    "mid": "frequency band roughly 0.5–3 Hz",
    "high":"frequency band roughly larger than 3 Hz",
    "ratio":"(band power)/(total power)",
    "ent":"spectral entropy",
    "centroid":"spectral centroid",
    "diff":"first-order difference",
    "zc_rate":"zero-crossing rate",
    "skew":"third statistical moment",
    "kurtosis":"fourth statistical moment",
    "grav_angle":"angle between a sensor axis and the low-pass filtered acceleration (gravity)",
    "dyn_angle":"angle between a sensor axis and the high-pass filtered acceleration (dynamic motion)",
    "sign": "the sign of the average dynamic acceleration over the window, indicating net negative, neutral, or net positive motion direction",
    "mav":"mean absolute value",
    "sav":"signal magnitude area (sum of abs values)",
    "fft":"fast Fourier transform (frequency-domain)",
    "stft":"short-time Fourier transform (time-freq spectrogram)",
    "acf_first_peak_lag":"lag of first ACF local maximum (>0)",
    "acf_first_min_lag":"lag of first ACF local minimum (>0)",
    "acf_first_zero_lag":"lag where ACF first crosses zero",
    "jerk":"first derivative of the signal (diff/Δt)",
    "intensity_ratio_acc_gyro":"ratio of RMS(acc) to RMS(gyro)",
    "corr":"Pearson correlation between two channels",
    # —— Burg AR(4) coefficients ——
    "ar1":     "first AR(4) coefficient a₁ estimated by Burg’s method",
    "ar2":     "second AR(4) coefficient a₂ estimated by Burg’s method",
    "ar3":     "third AR(4) coefficient a₃ estimated by Burg’s method",
    "ar4":     "fourth AR(4) coefficient a₄ estimated by Burg’s method",
    # —— Burg AR(4) model variance ——
    "ar_var":  "prediction error variance of the Burg AR(4) model",
    "sms": "Signal Magnitude Area",
    "fft_max_idx": "index of max magnitude in PSD",
    "weighted_avg": "weighted average",
    "energy": "spectral energy",
    "rr": "recurrence rate",
    "wpd_L": "the n-th level of the wavelet packet decomposition",
    "pe": "permutation entropy"
}


## Prompt 1: Feature Selection

System and user prompts for selecting discriminative features from pairwise priors.


In [18]:
# Step 1: choose discriminative features

choose_features_prompt_sys = """
Task
----
You're an expert in Human Activity Recognition, with a focus on identifying the most effective features for distinguishing between human activities.

Glossary of Abbreviations
-------------------------
{GLOSS_TEXT}

Instructions
------------
1. Based on the user-provided “Top Features per Activity Pair“, select up to {TOP_N} unique features that best distinguish the specified Target Activities.
2. When selecting features, prioritize those that:
    • Appear consistently across multiple activity pairs, or
    • Have relatively high importance scores within specific pairs.
3. For each selected feature, give:
   • *Definition* – A concise, clear explanation of the feature.
   • *Reason* – Summarize the following:
        1. Which activity pairs this feature helps to distinguish.
        2. A brief justification explaining why this feature effectively differentiates those classes.

Output Format
-------------
Return only the following, in this exact order, with no extra text or line breaks:

A plain-text markdown table with the following columns:
| Index | Feature_name | Definition | Reason |

A JSON array of the chosen feature names, using their original spelling:
["feature_1", "feature_2", …]
"""

choose_features_prompt = """
Target Activities
-----------------
{ACTIVITY_LIST}

Top Features per Activity Pair (Ranked by Importance Score)
({PAIR_NUM} activity pairs in total)
-----------------------------------------------------------
{FEATURE_IMPORTANCE}
"""


## Prompt 2: Activity Narrowing

Prompts for narrowing the candidate activity set using query/reference feature tables.


In [19]:
# Step 2: narrow down candidate activities

narrow_down_activities_sys = """
Task
----
You are an expert in Human Activity Recognition.
Your task is to narrow down the set of plausible activities for the **QUERY** segment, based on the given features and their statistical distributions in the user-supplied activities table.

Instructions
------------
- Each row in the activities table represents an activity class, with cells showing the mean ± std of each feature.
- The **QUERY** row presents the feature values of the segment to classify.
- You must select **at least two** activity classes, but may include more if they are reasonably or even marginally plausible for the QUERY.
- For each selected activity class, provide a brief explanation of why it is a plausible match.

Output Format
-------------
Only return the following, in this exact order, with no additional text or line breaks:

A plain-text markdown table with the following columns:
| Index | Activity | Reason |

A JSON array of the chosen activity class names, using their original spelling:
```json
["Activity_1", "Activity_2", ..., "Activity_k"]
```
"""

narrow_down_activities_prompt = """
Activities Table
----------------
{ACTIVITIES_FEATURES_TABLE}
"""


## Prompt 3: Feature Refinement

Prompts for refining features after the candidate set is narrowed.


In [20]:
# Step 3: refine features for filtered activities
choose_features_prompt_sys2 = """
Task
----
You're an expert in Human Activity Recognition, with a focus on identifying the most effective features for distinguishing between human activities.

Glossary of Abbreviations
-------------------------
{GLOSS_TEXT}

Instructions
------------
1. Based on the user-provided “Top Features per Activity Pair“, select up to {TOP_N} unique features that best distinguish the specified Target Activities.
2. When selecting features, prioritize those that:
    • Appear consistently across multiple activity pairs, or
    • Have relatively high importance scores within specific pairs.
3. For each selected feature, give:
   • *Definition* – A concise, clear explanation of the feature.
   • *Discriminative Power* – Summarize the following:
	1.	Which activity pairs this feature helps to distinguish.
	2.	The relative importance of the feature within each activity pair reflects how well it distinguishes the activities within that pair.

Output Format
-------------
Return only the following, in this exact order, with no extra text or line breaks:

A plain-text markdown table with the following columns:
| Index | Feature_name | Definition | Discriminative Power |

A JSON array of the chosen feature names, using their original spelling:
["feature_1", "feature_2", …]
"""

choose_features_prompt2 = """
Target Activities
-----------------
{ACTIVITY_LIST}

Top Features per Activity Pair (Ranked by Importance Score)
({PAIR_NUM} activity pairs in total)
-----------------------------------------------------------
{FEATURE_IMPORTANCE}
"""


## Prompt 4: Final Classification

Prompts for final activity classification and rationale generation.


In [21]:
# Step 4: classify among filtered activities

choose_activity_prompt_sys = """
Task
----
You are an expert in Human Activity Recognition (HAR).
The user is equipped with two 3-axis sensors: an accelerometer and a gyroscope.
Your task is to determine the most probable activity class of the **Query** segment by comparing its sensor-derived features with those of labeled activity examples.

Instructions
------------
- The **last row**, labeled **QUERY**, represents the data segment to classify.
- Each row contains features extracted from a time window of sensor data.
- Choose the **single most likely activity class** for the Query row.

Requirements for Your Decision:
- First, explicitly compare the Query’s features with those of each other activity class individually, rather than making general comparisons. Show why the predicted class is more likely than each alternative.
"""

choose_activity_mean_prompt_sys = """
Task
----
You are an expert in Human Activity Recognition.
Your goal is to determine the **most probable activity class** for the **QUERY** segment by comparing its feature values against the statistical distributions in the user-provided activities table.

Sensor Feature Explanation Guide Table
--------------------------------------
This table describes each feature and indicates which activity classes it helps to distinguish between.
{FEATURES_REF_TABLE}

Instructions
------------
- Each row in the activities table corresponds to an activity class, with each cell showing the mean ± standard deviation for a feature.
- The **QUERY** row presents the feature values of the segment to classify.
- Select the **single most likely activity class**, and base your decision on **specific feature(s)** in the QUERY row.
- In your explanation:
  1. Explicitly compare the Query’s feature values to each class’s distribution, explaining why the predicted class is a better match than each alternative.
  2. When unsure, refer to the Discriminative Power in the guide table to justify how strongly each feature helps distinguish the specific activities.

Output Format
--------------
**Respond with exactly one line** in this JSON format (no extra text or line breaks):
```json
{{"reason":"<your detailed explanation>", "predicted_class":"<ClassName>"}}
```
"""

choose_activity_prompt = """
Activities Table
----------------
{ACTIVITIES_FEATURES_TABLE}
"""


## Inference Utilities

Define embedding, ranking, prompt-building, Gemini retry, and table-formatting helpers.


In [22]:
import time
from google.genai.errors import ServerError
from collections import defaultdict

def calculate_emb(data_np, model, device):
    """Compute query embeddings with the frozen time-series encoder."""
    data_torch = torch.tensor([data_np], dtype=torch.float32, device=device)  # (1, num_channels, seq_len)
    data_torch_scaled = F.interpolate(data_torch, size=512, mode='linear', align_corners=False)
    assert data_torch_scaled.shape[1] == 6

    tl_embeddings_np = model.transform(data_torch_scaled)

    return tl_embeddings_np

def sort_by_activity(fused_doc_ids, fused_labels, fused_scores, id2label, top_activity_num):
    """Group retrieved examples by activity and keep the top-ranked candidate activities."""
    from collections import defaultdict

    # 1) collect per‐label lists
    label_dict = defaultdict(list)
    for db_idx, label, sim in zip(fused_doc_ids, fused_labels, fused_scores):
        label_dict[label].append((db_idx, sim))

    # 2) sort each list by sim desc
    for label in label_dict:
        label_dict[label].sort(key=lambda x: x[1], reverse=True)

    # 3) rank labels by their top‐sim score
    ranked = sorted(
        label_dict.keys(),
        key=lambda l: label_dict[l][0][1],
        reverse=True
    )

    # 4) keep only the top N activities
    top_labels = ranked[:top_activity_num]

    # 5) now sort those top labels alphabetically by their name
    top_labels_sorted = sorted(top_labels, key=lambda l: id2label[l])

    # 6) build your outputs in that alphabetical order
    trimmed_label_dict = {l: label_dict[l] for l in top_labels_sorted}
    label_names        = [id2label[l] for l in top_labels_sorted]

    return trimmed_label_dict, label_names

def build_pairwise_prompt_and_list(activity_list, pair_map, top_n=15):
    """Build the pairwise feature-importance prompt and its feature list."""
    lines = []
    seen = []
    seen_set = set()

    for a, b in combinations(activity_list, 2):
        fmap = pair_map.get((a, b)) or pair_map.get((b, a), {})
        valid_items = [(f, w) for f, w in fmap.items() if w > 0]
        if not valid_items:
            continue
        top_feats = sorted(valid_items, key=lambda x: x[1], reverse=True)[:top_n]

        entry = f"{a} vs {b}:"
        for feat, w in top_feats:
            entry += f"\n   • {feat} ({round(w*100, 2)}%)"
            if feat not in seen_set:
                seen_set.add(feat)
                seen.append(feat)
        lines.append(entry)

    prompt_str = "\n\n".join(lines)
    unique_feats = sorted(seen)
    return prompt_str, unique_feats

def call_gemini(client, sys_prompt, prompt):
    """Send one prompt to Gemini and return the text response."""
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[prompt],
        config=types.GenerateContentConfig(
            system_instruction=sys_prompt,
            temperature=0,
        )
    )
    return response.text


def call_gemini_with_retry(client, sys_prompt, prompt,
                           max_retry=10,
                           base_delay=1.0,
                           max_delay=30.0):
    """Call Gemini with exponential backoff for transient service errors."""
    attempt = 0
    while True:
        try:
            return call_gemini(client, sys_prompt, prompt)
        except ServerError as e:
            if e.code == 503 or getattr(e, "status", "") == "UNAVAILABLE":
                attempt += 1
                if attempt > max_retry:
                    raise RuntimeError(f"Exceeded maximum retries ({max_retry}): {e}")
                delay = min(base_delay * 2 ** (attempt - 1), max_delay)
                print(f"[Retry {attempt}/{max_retry}] 503 model overloaded, sleeping {delay:.1f}s ...")
                time.sleep(delay)
                continue
            raise


def to_row(d, columns):
    """Format one dictionary as a Markdown table row."""
    cells = [str(d.get(col, "")) for col in columns]
    return "| " + " | ".join(cells) + " |\n"


def form_features_mkd_table(feature_list, query_feats, db_feats_df, label_dict, R_NUM, use_summary=False, label_names=None, N1=4, N2=2):
    """Format query and reference feature values as a Markdown table for prompting."""
    query_row = {k: f"{query_feats[k]:.{N1}f}" for k in feature_list}
    table_rows = []

    for lab, refs in label_dict.items():
        label_name = id2label[lab]
        if label_names is not None and label_name not in label_names:
            continue

        mat = []
        for db_idx, _ in refs[:R_NUM]:
            feats_all = db_feats_df.iloc[db_idx].to_dict()
            mat.append([feats_all[k] for k in feature_list])
            if not use_summary:
                table_rows.append(
                    {"label": label_name, **{k: f"{feats_all[k]:.{N1}f}" for k in feature_list}}
                )

        if use_summary and mat:
            arr = np.array(mat)
            means = arr.mean(axis=0)
            stds = arr.std(axis=0)
            summary = {k: f"{m:.{N1}f}±{s:.{N2}f}" for k, m, s in zip(feature_list, means, stds)}
            table_rows.append({"label": label_name, **summary})

    columns = ["label"] + feature_list
    header = "| " + " | ".join(columns) + " |\n"
    sepline = "|---" * len(columns) + "|\n"
    md_lines = [header, sepline]

    for row in table_rows:
        md_lines.append(to_row(row, columns))
    md_lines.append(to_row({"label": "QUERY", **query_row}, columns))

    return "".join(md_lines)


def is_flat_list_of_scalars(obj):
    """Check that a parsed JSON value is a flat list of scalar entries."""
    return isinstance(obj, list) and all(not isinstance(el, (dict, list, tuple, set)) for el in obj)


def safe_fix_json_text(resp):
    """Repair common malformed JSON formatting in model responses."""
    if not isinstance(resp, (str, bytes)):
        text = getattr(resp, "text", None)
        text = text if isinstance(text, (str, bytes)) else str(resp)
    else:
        text = resp

    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)

    pattern = r'("reason"\s*:\s*")(.+?)(?="(,|\s*\}))'

    def _escape(match):
        """Escape problematic characters inside a JSON string field."""
        body = match.group(2)
        body = body.replace('\\', '\\\\').replace('"', '\\"')
        body = body.replace('\n', '\\n').replace('\r', '')
        return f'{match.group(1)}{body}'

    return re.sub(pattern, _escape, text, flags=re.DOTALL)


## Run ZARA Inference

Retrieve evidence, select features, narrow activities, and produce final predictions for each query.


In [23]:
TOTAL_K = len(all_database_labels)
SIM_DIFF = 1
TOP_N = 10
TOP_N2 = 10
R_FEATURES_NUM = 25
R_FEATURES_NUM2 = 5
TOP_ACTIVITY_NUM = 6
pair_num1 = int(TOP_ACTIVITY_NUM*(TOP_ACTIVITY_NUM-1)/2)
DEVICE="mps"

R_DATA_NUM = 100
USE_SMRY=True

md_table_pattern = r"(\|.*\|[\r\n]+(\|.*\|[\r\n]+)+)"
json_pattern = r"\[[^\]]+\]"

acc = 0
narrow_acc = 0

output_dict = []

print("****"*10)
print(f"choose_features_prompt_sys:\n{choose_features_prompt_sys}")
print("****"*10)
print(f"narrow_down_activities_sys:\n{narrow_down_activities_sys}")
print("****"*10)
print(f"choose_features_prompt_sys2:\n{choose_features_prompt_sys2}")
print("****"*10)
print(f"choose_activity_mean_prompt_sys:\n{choose_activity_mean_prompt_sys}")
print("****"*10)

print(f"For each activity-pair (totally {pair_num1}), show top {TOP_N} important features to choose.")
print(f"Ask LLM to choose up to {R_FEATURES_NUM}/{R_FEATURES_NUM2} features to classify.")
print(f"Input {R_DATA_NUM} examples each class to LLM for reference.")
print(f"Input {TOP_ACTIVITY_NUM} activities' data to LLM for reference.")

choose_features_response = None
md_table = None
unique_list = None
prompt_str = None

for i, (t, l) in enumerate(zip(all_test_segments, all_test_labels)):
    print(f"Processed Query {i}")

    t = np.array(t)
    t = t.transpose(1, 0)  # (num_channels, seq_len)
    ground_truth = l['activity_name']
    # Compute the query embedding.
    tl_embeddings_np = calculate_emb(t, model, DEVICE)
    # Retrieve from the database index.
    sensor_topk_sims, sensor_topk_indices = query_faiss(index_sensor, tl_embeddings_np, TOTAL_K, sim_diff=SIM_DIFF)
    # Decode retrieval labels.
    sensor_topk_labels = [all_database_labels[i]['activity'] for i in sensor_topk_indices]
    # Build the feature comparison table.
    query_features = extract_features(t)

    label_dict, label_names = sort_by_activity(sensor_topk_indices, sensor_topk_labels, sensor_topk_sims, id2label, TOP_ACTIVITY_NUM)

    print(f"Input activities: {label_names}")

    # Step 1: Select Top-N Discriminative Features for All Target Activities
    if choose_features_response is None:
        prompt_str, unique_feats = build_pairwise_prompt_and_list(label_names, pair_map, top_n=TOP_N)
        # Build the glossary for the selected pairwise features.
        sub_glossary = extract_glossary_for_features(unique_feats, full_glossary)
        # Format the feature glossary for the prompt.
        gloss_lines = []
        for token, definition in sub_glossary.items():
            gloss_lines.append(f"• {token}: {definition}")
        gloss_text = "\n".join(gloss_lines)

        max_retry = 10
        attempt = 0
        while attempt < max_retry:
            try:
                choose_features_response = call_gemini_with_retry(
                    client,
                    choose_features_prompt_sys.format(GLOSS_TEXT=gloss_text, TOP_N=R_FEATURES_NUM),
                    choose_features_prompt.format(ACTIVITY_LIST=label_names, FEATURE_IMPORTANCE=prompt_str, PAIR_NUM=pair_num1)
                )

                # Parse the Markdown table.
                md_match = re.search(md_table_pattern, choose_features_response)
                if not md_match:
                    raise ValueError("Markdown table not found!")
                md_table = md_match.group(1).strip()

                # Parse the JSON list.
                json_match = re.search(json_pattern, choose_features_response)
                if not json_match:
                    raise ValueError("JSON array not found!")
                feature_list = json.loads(json_match.group(0))

                unique_list  = []
                for item in feature_list:
                    if item not in unique_list:
                        unique_list.append(item)
                assert len(unique_list) <= R_FEATURES_NUM, f"unique_list length {len(unique_list)} not fit R_FEATURES_NUM {R_FEATURES_NUM}"

                missing = [k for k in unique_list if k not in query_features]
                if missing:
                    raise KeyError(f"Query is missing these requested features: {missing}")

                break

            except (json.JSONDecodeError, AssertionError, Exception) as e:
                attempt += 1
                print(f"[Retry {attempt}/{max_retry}] Failed to parse Gemini response: {choose_features_response}")
                if attempt == max_retry:
                    raise RuntimeError(f"Exceeded maximum retries ({max_retry}) for choose_features_response.")

    cls_markdown_table = form_features_mkd_table(unique_list, query_features, all_database_features_df, label_dict, R_DATA_NUM, USE_SMRY)

    # Step 2: Narrow Down Possible Activities Based on Query
    max_retry = 20
    attempt = 0
    narrow_down_activities_response=None
    narrow_down_md_table = None
    narrow_down_label_list = None
    while attempt < max_retry:
        try:
            narrow_down_activities_response = call_gemini_with_retry(
                client,
                narrow_down_activities_sys,
                narrow_down_activities_prompt.format(ACTIVITIES_FEATURES_TABLE=cls_markdown_table)
            )
             # Parse the Markdown table.
            narrow_down_md_match = re.search(md_table_pattern, narrow_down_activities_response)
            if not narrow_down_md_match:
                raise ValueError("Markdown table not found!")
            narrow_down_md_table = narrow_down_md_match.group(1).strip()

            # Parse the JSON list.
            narrow_down_json_match = re.search(json_pattern, narrow_down_activities_response)
            if not narrow_down_json_match:
                raise ValueError("JSON array not found!")
            narrow_down_label_list = json.loads(narrow_down_json_match.group(0))

            assert len(narrow_down_label_list) >= 2
            assert is_flat_list_of_scalars(narrow_down_label_list)

            break

        except (json.JSONDecodeError, AssertionError, Exception) as e:
            attempt += 1
            print(f"[Retry {attempt}/{max_retry}] Failed to parse Gemini response: {narrow_down_activities_response}")
            if attempt == max_retry:
                raise RuntimeError(f"Exceeded maximum retries ({max_retry}) for choose_features_response.")

    print(f"Narrow-down activities: {narrow_down_label_list}")

    activity_json = {}
    if ground_truth.lower() not in [item.lower() for item in narrow_down_label_list]:
        if i == 0:
            print(f"Input 1: {prompt_str}")
            print("--------")
            print(f"LLM Output 1: {md_table}\n\nunique list number:{len(unique_list)}\n{unique_list}")
            print("--------")
            print(f"Input 2: {cls_markdown_table}")
            print("--------")
            print(f"LLM Output 2: {narrow_down_md_table}\n\n{narrow_down_label_list}")
            print("--------"*5)
        activity_json["predicted_class"] = ""
        activity_json["ground_truth"] = ground_truth
        activity_json["id"] = i
        activity_json["LLM_selected_features_md_table"] = md_table
        activity_json["LLM_selected_features_list"] = unique_list
        activity_json["activity_features_md_table"] = cls_markdown_table
        activity_json["LLM_narrow_down_md_table"] = narrow_down_md_table
        activity_json["LLM_narrow_down_label_list"] = narrow_down_label_list
        i += 1
        print(f"Ground-truth: {ground_truth}\nnarrow down list accuracy: {narrow_acc/i}\naccuracy: {acc/i}")
        print("-------"*10)
        output_dict.append(activity_json)
        continue

    # Step 3: Select New Top-O Features for Filtered Activities
    max_retry = 20
    attempt = 0
    rechoose_features_response = None
    rechoose_md_table = None
    unique_rechoose_feature_list = None

    rechoose_prompt_str, rechoose_unique_feats = build_pairwise_prompt_and_list(narrow_down_label_list, pair_map, top_n=TOP_N2)
    rechoose_sub_glossary = extract_glossary_for_features(rechoose_unique_feats, full_glossary)
    # Format the feature glossary for the prompt.
    rechoose_gloss_lines = []
    for token, definition in rechoose_sub_glossary.items():
        rechoose_gloss_lines.append(f"• {token}: {definition}")
    rechoose_gloss_text = "\n".join(rechoose_gloss_lines)

    while attempt < max_retry:
        try:
            pair_num = int(len(narrow_down_label_list)*(len(narrow_down_label_list)-1)/2)
            rechoose_num =int(pair_num*R_FEATURES_NUM2)
            rechoose_features_response = call_gemini_with_retry(
                client,
                choose_features_prompt_sys2.format(GLOSS_TEXT=rechoose_gloss_text, TOP_N=rechoose_num),
                choose_features_prompt2.format(ACTIVITY_LIST=narrow_down_label_list, FEATURE_IMPORTANCE=rechoose_prompt_str, PAIR_NUM=pair_num)
            )

            # Parse the Markdown table.
            rechoose_md_match = re.search(md_table_pattern, rechoose_features_response)
            if not rechoose_md_match:
                raise ValueError("Markdown table not found!")
            rechoose_md_table = rechoose_md_match.group(1).strip()

            # Parse the JSON list.
            rechoose_json_match = re.search(json_pattern, rechoose_features_response)
            if not rechoose_json_match:
                raise ValueError("JSON array not found!")
            rechoose_feature_list = json.loads(rechoose_json_match.group(0))

            unique_rechoose_feature_list = []
            for item in rechoose_feature_list:
                if item not in unique_rechoose_feature_list:
                    unique_rechoose_feature_list.append(item)

            assert len(unique_rechoose_feature_list) <= rechoose_num, f"unique_rechoose_feature_list length {len(unique_rechoose_feature_list)} not fit rechoose_num {rechoose_num}"

            missing = [k for k in unique_rechoose_feature_list if k not in query_features]
            if missing:
                raise KeyError(f"Query is missing these requested features: {missing}")

            break

        except (json.JSONDecodeError, AssertionError, Exception) as e:
            attempt += 1
            print(f"[Retry {attempt}/{max_retry}] Failed to parse Gemini response: {rechoose_features_response}")
            if attempt == max_retry:
                raise RuntimeError(f"Exceeded maximum retries ({max_retry}) for choose_features_response.")

    rechoose_cls_markdown_table = form_features_mkd_table(unique_rechoose_feature_list, query_features, all_database_features_df, label_dict, R_DATA_NUM, USE_SMRY, narrow_down_label_list)

    # Step 4: Final Classification Among Filtered Activities
    max_retry = 20
    attempt = 0
    choose_activity_response = None
    while attempt < max_retry:
        try:
            if USE_SMRY:
                choose_activity_response = call_gemini_with_retry(
                        client,
                        choose_activity_mean_prompt_sys.format(FEATURES_REF_TABLE=rechoose_md_table),
                        choose_activity_prompt.format(ACTIVITIES_FEATURES_TABLE=rechoose_cls_markdown_table)
                    )
            else:
                choose_activity_response = call_gemini_with_retry(
                    client,
                    choose_activity_prompt_sys.format(FEATURES_REF_TABLE=rechoose_md_table),
                    choose_activity_prompt.format(ACTIVITIES_FEATURES_TABLE=rechoose_cls_markdown_table)
                )

            # Extract the JSON object.
            m = re.search(r'\{.*\}', choose_activity_response.strip(), re.DOTALL)
            if not m:
                raise ValueError(f"No JSON object found in LLM output:\n{choose_activity_response.strip()}")
            json_str = m.group(0)

            if i == 0:
                print(f"Input 1: {prompt_str}")
                print("--------")
                print(f"LLM Output 1: {md_table}\n\nunique list number:{len(unique_list)}\n{unique_list}")
                print("--------")
                print(f"Input 2: {cls_markdown_table}")
                print("--------")
                print(f"LLM Output 2: {narrow_down_md_table}\n\n{narrow_down_label_list}")
                print("--------")
                print(f"Input 3: {rechoose_prompt_str}")
                print("--------")
                print(f"LLM Output 3: {rechoose_md_table}\n\nunique_rechoose_feature_list number:{len(unique_rechoose_feature_list)}\n{unique_rechoose_feature_list}")
                print("--------")
                print(f"Input 4: {rechoose_cls_markdown_table}")
                print("--------")
                print(f"LLM Output 4: {json_str}")
                print("--------"*5)

            # Load the JSON object.
            activity_json = json.loads(json_str)

            assert "predicted_class" in activity_json, "predicted_class not in activity_json"
            assert activity_json['predicted_class'].lower() in  [item.lower() for item in label_names], "predicted_class not in input labels"

            break

        except (json.JSONDecodeError, AssertionError, Exception) as e:
            attempt += 1
            print(f"[Retry {attempt}/{max_retry}]  Failed to parse activity prediction from LLM response")
            if attempt == max_retry:
                raise RuntimeError(f"Exceeded maximum retries ({max_retry}) for choose_features_response.")

            # Repair malformed JSON text from the full LLM response.
            fixed_response = safe_fix_json_text(choose_activity_response)

            # Extract and parse the repaired JSON object.
            m2 = re.search(r'\{.*?\}', fixed_response.strip(), re.DOTALL)
            if not m2:
                print(f"Still no JSON after fix:\n{choose_activity_response}, retrying...")
                continue

            fixed_json_str = m2.group(0)
            try:
                activity_json = json.loads(fixed_json_str)
                assert "predicted_class" in activity_json, "predicted_class not in activity_json"
                assert activity_json['predicted_class'].lower() in  [item.lower() for item in label_names], "predicted_class not in input labels"
                break
            except json.JSONDecodeError as e2:
                time.sleep(2.0)
                print(f"Failed to parse fixed JSON: {e2}. Retrying...")

    activity_json["ground_truth"] = ground_truth
    activity_json["id"] = i
    activity_json["LLM_selected_features_md_table"] = md_table
    activity_json["LLM_selected_features_list"] = unique_list
    activity_json["activity_features_md_table"] = cls_markdown_table
    activity_json["LLM_narrow_down_md_table"] = narrow_down_md_table
    activity_json["LLM_narrow_down_label_list"] = narrow_down_label_list
    activity_json["LLM_rechoose_md_table"] = rechoose_md_table
    activity_json["LLM_rechoose_feature_list"] = unique_rechoose_feature_list
    activity_json["activity_rechoose_features_md_table"] = rechoose_cls_markdown_table

    i += 1
    narrow_acc += 1
    if ground_truth.lower() == activity_json['predicted_class'].lower():
        acc +=1

    print(f"Ground-truth: {ground_truth}\nPredicted_class: {activity_json['predicted_class']}")

    output_dict.append(activity_json)

    print(f"narrow down list accuracy: {narrow_acc/i}\naccuracy: {acc/i}")

    print("-------"*10)


****************************************
choose_features_prompt_sys:

Task
----
You're an expert in Human Activity Recognition, with a focus on identifying the most effective features for distinguishing between human activities.

Glossary of Abbreviations
-------------------------
{GLOSS_TEXT}

Instructions
------------
1. Based on the user-provided “Top Features per Activity Pair“, select up to {TOP_N} unique features that best distinguish the specified Target Activities.
2. When selecting features, prioritize those that:
    • Appear consistently across multiple activity pairs, or
    • Have relatively high importance scores within specific pairs.
3. For each selected feature, give:
   • *Definition* – A concise, clear explanation of the feature.
   • *Reason* – Summarize the following:
        1. Which activity pairs this feature helps to distinguish.
        2. A brief justification explaining why this feature effectively differentiates those classes.

Output Format
-------------
R

/var/folders/fn/tqqbvv9x7fj7jthvdm9f8m000000gn/T/ipykernel_50632/1594579110.py:94: ComplexWarning: Casting complex values to real discards the imaginary part
  feats[f"{name}_{axis}_ar{k}"] = float(ar_coeffs[k])


Input activities: ['Laying', 'Sitting', 'Standing', 'Walking', 'Walking downstairs', 'Walking upstairs']
Narrow-down activities: ['Standing', 'Sitting']
Input 1: Laying vs Sitting:
   • acc_x_mean (76.07%)
   • acc_x_min (18.56%)
   • acc_mag_bp_fft_mid (1.12%)
   • acc_x_rms (0.76%)
   • acc_x_stft_high_max (0.76%)
   • acc_x_max (0.7%)
   • acc_x_stft_mid_max (0.68%)
   • acc_x_stft_mid_mean (0.68%)
   • acc_x_stft_mid_std (0.68%)

Laying vs Standing:
   • acc_x_mean (49.02%)
   • acc_x_min (41.84%)
   • acc_y_max (9.14%)

Laying vs Walking:
   • acc_x_mean (42.24%)
   • acc_x_max (31.41%)
   • acc_x_std (26.36%)

Laying vs Walking downstairs:
   • acc_mag_bp_fft_low_ratio (26.73%)
   • acc_mag_wpd_L1_entropy (26.73%)
   • acc_mag_max (11.79%)
   • acc_x_median (6.68%)
   • acc_x_max (6.06%)
   • acc_mag_jerk_rms (5.85%)
   • acc_mag_range (5.85%)
   • acc_x_acf_first_zero_lag (5.08%)
   • acc_x_skew (2.83%)
   • acc_x_min (0.87%)

Laying vs Walking upstairs:
   • acc_mag_wpd_L1_entr

## Save Predictions

Persist the structured prediction output for later evaluation or inspection.


In [24]:
with open(os.path.join("./predictions", f"prediction_uci.json"), "w", encoding="utf-8") as f:
    json.dump(output_dict, f, ensure_ascii=False, indent=2)
